In [ ]:
from pathlib import Path
from dotenv import dotenv_values

import requests

import pandas as pd
import plotly.express as px

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent

from sqlalchemy import (
    BigInteger,
    Boolean,
    Column,
    DateTime,
    Float,
    MetaData,
    String,
    Table,
    Text,
    create_engine,
    inspect,
)
from sqlalchemy.engine import URL
from sqlalchemy.schema import CreateSchema
from sqlalchemy import Integer, MetaData, Table, func, select


In [ ]:
env = dotenv_values(project_root / ".env")
try:
    host = "localhost"
    port = int(env.get("POSTGRES_PORT", 5432))# type:ignore
    database = env["POSTGRES_DB"]
    user = env["POSTGRES_USER"]
    password = env["POSTGRES_PASSWORD"]
    schema_name = "public"
    table_name = "pubseg"
except:
    raise

In [ ]:
def build_postgres_engine(host: str, port: int, database: str, user: str, password: str):
    return create_engine(
        URL.create(
            "postgresql+psycopg2",
            username=user,
            password=password,
            host=host,
            port=port,
            database=database,
        )
    )


In [ ]:
engine = build_postgres_engine(host, port, database, user, password)
metadata_public = MetaData(schema=schema_name)
pubseg = Table(table_name, metadata_public, autoload_with=engine, schema=schema_name)

In [ ]:
for column in pubseg.columns:
    print(f"{column.name}: {column.type}")


# Análise exploratória

Plot gerado pelo Codex

## Incidência e Boxplot por evento

In [ ]:
import plotly.graph_objects as go

victim_columns = [
    ("masculino", "Masculino"),
    ("feminino", "Feminino"),
    ("nao_informado", "Não informado"),
    ("total_vitima", "Total de vítimas"),
]

eventos_disponiveis = [
    "Apreensão de Cocaína",
    "Apreensão de Maconha",
    "Arma de Fogo Apreendida",
    "Atendimento pré-hospitalar",
    "Busca e salvamento",
    "Combate a incêndios",
    "Emissão de Alvarás de licença",
    "Estupro",
    "Estupro de vulnerável",
    "Feminicídio",
    "Furto de veículo",
    "Homicídio doloso",
    "Lesão corporal seguida de morte",
    "Mandado de prisão cumprido",
    "Morte de Agente do Estado",
    "Morte no trânsito ou em decorrência dele (exceto homicídio doloso)",
    "Morte por intervenção de Agente do Estado",
    "Mortes a esclarecer (sem indício de crime)",
    "Mortes no trânsito",
    "Pessoa Desaparecida",
    "Pessoa Localizada",
    "Realização de vistorias",
    "Roubo a instituição financeira",
    "Roubo de carga",
    "Roubo de veículo",
    "Roubo seguido de morte (latrocínio)",
    "Suicídio",
    "Suicídio de Agente do Estado",
    "Tentativa de feminicídio",
    "Tentativa de homicídio",
    "Tráfico de drogas",
]


def build_event_filter(evento: str | None) -> list:
    if evento is None:
        return []

    return [pubseg.c.evento == evento]


def build_victim_quantile_summary(evento: str | None) -> pd.DataFrame:
    summary_rows = []
    event_filter = build_event_filter(evento)

    with engine.connect() as conn:
        for column_name, column_label in victim_columns:
            column = getattr(pubseg.c, column_name)
            stats = conn.execute(
                select(
                    func.min(column).label("minimo"),
                    func.percentile_cont(0.25).within_group(column).label("q1"),
                    func.percentile_cont(0.5).within_group(column).label("mediana"),
                    func.percentile_cont(0.75).within_group(column).label("q3"),
                    func.max(column).label("maximo"),
                ).where(
                    column.is_not(None),
                    *event_filter,
                )
            ).mappings().one()

            summary_rows.append({
                "coluna": column_label,
                "minimo": stats["minimo"],
                "q1": stats["q1"],
                "mediana": stats["mediana"],
                "q3": stats["q3"],
                "maximo": stats["maximo"],
            })

    return pd.DataFrame(summary_rows)


def plot_victim_incidence_by_event(evento: str | None) -> None:
    event_filter = build_event_filter(evento)
    event_suffix = f" para {evento}" if evento is not None else ""

    with engine.connect() as conn:
        max_total_vitima = conn.execute(
            select(func.max(pubseg.c.total_vitima)).where(
                pubseg.c.total_vitima.is_not(None),
                *event_filter,
            )
        ).scalar_one()

        if max_total_vitima is None:
            print(f"Nenhum dado encontrado para o evento: {evento}")
            return

        for column_name, column_label in victim_columns:
            column = getattr(pubseg.c, column_name)
            incidence_stmt = (
                select(
                    column.label("numero_vitimas"),
                    func.count().label("incidencia"),
                )
                .where(
                    column.is_not(None),
                    *event_filter,
                )
                .group_by(column)
                .order_by(column)
            )
            incidence_df = pd.read_sql(incidence_stmt, conn)

            stats = conn.execute(
                select(
                    func.avg(column).label("media"),
                    func.var_samp(column).label("variancia"),
                ).where(
                    column.is_not(None),
                    *event_filter,
                )
            ).mappings().one()

            print(f"{column_label}{event_suffix}")
            print(f"Média: {stats['media']}")
            print(f"Variância: {stats['variancia']}")
            print()

            fig = px.bar(
                incidence_df,
                x="numero_vitimas",
                y="incidencia",
                title=f"Incidência de {column_label}{event_suffix}",
                labels={
                    "numero_vitimas": "Número de vítimas",
                    "incidencia": "Quantidade de ocorrências",
                },
            )
            fig.update_xaxes(range=[0, max_total_vitima])
            fig.show()


def plot_victim_boxplot_by_event(evento: str | None) -> None:
    summary_df = build_victim_quantile_summary(evento)

    if summary_df[["minimo", "q1", "mediana", "q3", "maximo"]].isna().all().all():
        print(f"Nenhum dado encontrado para o evento: {evento}")
        return

    display(summary_df)

    event_suffix = f" para {evento}" if evento is not None else ""
    fig = go.Figure()
    for row in summary_df.itertuples(index=False):
        fig.add_trace(
            go.Box(
                name=row.coluna,
                lowerfence=[row.minimo],
                q1=[row.q1],
                median=[row.mediana],
                q3=[row.q3],
                upperfence=[row.maximo],
                boxpoints=False,
            )
        )

    fig.update_layout(
        title=f"Boxplot otimizado das colunas de vítimas{event_suffix}",
        yaxis_title="Número de vítimas",
    )
    fig.show()


### Apreensão de Cocaína
Como esperado Apreensão de Drogas não apresenta vitimas


In [ ]:
# Sem vitimas, comentado para limpar o notebook

# plot_victim_incidence_by_event("Apreensão de Cocaína")

# plot_victim_boxplot_by_event("Apreensão de Cocaína")


### Apreensão de Maconha
Como esperado Apreensão de Drogas não apresenta vitimas


In [ ]:
## Sem vitimas, comentado para limpar o notebook
#  plot_victim_incidence_by_event("Apreensão de Maconha")

# plot_victim_boxplot_by_event("Apreensão de Maconha")



### Arma de Fogo Apreendida
Apenas outra apreensão, não haver vitimas é esperado


In [ ]:
# Sem vitimas, comentado para limpar o notebook
# plot_victim_incidence_by_event("Arma de Fogo Apreendida")

# plot_victim_boxplot_by_event("Arma de Fogo Apreendida") 


### Atendimento pré-hospitalar
Não constam vítimas, será que em caso de óbito é classificado diferente ou simplismente não contam como vitimas?

In [ ]:
# Sem vitimas, comentado para limpar o notebook

# plot_victim_incidence_by_event("Atendimento pré-hospitalar")

# plot_victim_boxplot_by_event("Atendimento pré-hospitalar") 


### Busca e salvamento
O mesmo caso de antes, será que se houver vitimas é classificado diferente ou mesmo que haja óbito ão é considerado vítima?

In [ ]:
# Sem vitimas, comentado para limpar o notebook
# plot_victim_incidence_by_event("Busca e salvamento")

# plot_victim_boxplot_by_event("Busca e salvamento") 


### Combate a incêndios

Chegando nesse ponto me parece que só são contabilizadas vitimas em crimes violentos

In [ ]:
## Sem vitimas, comentado para limpar o notebook
# 
#  plot_victim_incidence_by_event("Combate a incêndios")

# plot_victim_boxplot_by_event("Combate a incêndios") 



### Emissão de Alvarás de licença

Porque haveria alguma vitima?

In [ ]:
# Sem vitimas, comentado para limpar o notebook

# plot_victim_incidence_by_event("Emissão de Alvarás de licença")

# plot_victim_boxplot_by_event("Emissão de Alvarás de licença")


### Estupro

Aqui há algo que não bate estúpro SEMPRE tem um vítima, como podem haver 87 ocorrências de estupro sem vitimas (ver em total_vitimas)?
Já tem uma ocorrência para cumprimento de mandados de prisão


In [ ]:
plot_victim_incidence_by_event("Estupro")

plot_victim_boxplot_by_event("Estupro")


### Estupro de vulnerável

In [ ]:
plot_victim_incidence_by_event("Estupro de vulnerável")

plot_victim_boxplot_by_event("Estupro de vulnerável")


### Feminicídio

In [ ]:
plot_victim_incidence_by_event("Feminicídio")

plot_victim_boxplot_by_event("Feminicídio")


### Furto de veículo

In [ ]:
# Sem vitimas, comentado para limpar o notebook

# plot_victim_incidence_by_event("Furto de veículo")

# plot_victim_boxplot_by_event("Furto de veículo")


### Homicídio doloso

In [ ]:
plot_victim_incidence_by_event("Homicídio doloso")

plot_victim_boxplot_by_event("Homicídio doloso")


### Lesão corporal seguida de morte

In [ ]:
plot_victim_incidence_by_event("Lesão corporal seguida de morte")

plot_victim_boxplot_by_event("Lesão corporal seguida de morte")


### Mandado de prisão cumprido

In [ ]:
# Sem vitimas, comentado para limpar o notebook

# plot_victim_incidence_by_event("Mandado de prisão cumprido")

# plot_victim_boxplot_by_event("Mandado de prisão cumprido")


### Morte de Agente do Estado

In [ ]:
plot_victim_incidence_by_event("Morte de Agente do Estado")

plot_victim_boxplot_by_event("Morte de Agente do Estado")


### Morte no trânsito ou em decorrência dele (exceto homicídio doloso)

In [ ]:
plot_victim_incidence_by_event("Morte no trânsito ou em decorrência dele (exceto homicídio doloso)")

plot_victim_boxplot_by_event("Morte no trânsito ou em decorrência dele (exceto homicídio doloso)")


### Morte por intervenção de Agente do Estado

In [ ]:
plot_victim_incidence_by_event("Morte por intervenção de Agente do Estado")

plot_victim_boxplot_by_event("Morte por intervenção de Agente do Estado")


### Mortes a esclarecer (sem indício de crime)

In [ ]:
plot_victim_incidence_by_event("Mortes a esclarecer (sem indício de crime)")

plot_victim_boxplot_by_event("Mortes a esclarecer (sem indício de crime)")


### Mortes no trânsito

In [ ]:
plot_victim_incidence_by_event("Mortes no trânsito")

plot_victim_boxplot_by_event("Mortes no trânsito")


### Pessoa Desaparecida

In [ ]:
plot_victim_incidence_by_event("Pessoa Desaparecida")

plot_victim_boxplot_by_event("Pessoa Desaparecida")


### Pessoa Localizada

In [ ]:
plot_victim_incidence_by_event("Pessoa Localizada")

plot_victim_boxplot_by_event("Pessoa Localizada")


### Realização de vistorias

In [ ]:
# Sem vitimas, comentado para limpar o notebook
# plot_victim_incidence_by_event("Realização de vistorias")

# plot_victim_boxplot_by_event("Realização de vistorias")


### Roubo a instituição financeira

In [ ]:
# Sem vitimas, comentado para limpar o notebook

# plot_victim_incidence_by_event("Roubo a instituição financeira")

# plot_victim_boxplot_by_event("Roubo a instituição financeira")


### Roubo de carga

In [ ]:
# Sem vitimas, comentado para limpar o notebook
# plot_victim_incidence_by_event("Roubo de carga")

# plot_victim_boxplot_by_event("Roubo de carga")


### Roubo de veículo

In [ ]:
# Sem vitimas, comentado para limpar o notebook
# plot_victim_incidence_by_event("Roubo de veículo")

# plot_victim_boxplot_by_event("Roubo de veículo")


### Roubo seguido de morte (latrocínio)

In [ ]:
plot_victim_incidence_by_event("Roubo seguido de morte (latrocínio)")

plot_victim_boxplot_by_event("Roubo seguido de morte (latrocínio)")


### Suicídio

In [ ]:
plot_victim_incidence_by_event("Suicídio")

plot_victim_boxplot_by_event("Suicídio")


### Suicídio de Agente do Estado

In [ ]:
plot_victim_incidence_by_event("Suicídio de Agente do Estado")

plot_victim_boxplot_by_event("Suicídio de Agente do Estado")


### Tentativa de feminicídio

In [ ]:
plot_victim_incidence_by_event("Tentativa de feminicídio")

plot_victim_boxplot_by_event("Tentativa de feminicídio")


### Tentativa de homicídio

In [ ]:
plot_victim_incidence_by_event("Tentativa de homicídio")

plot_victim_boxplot_by_event("Tentativa de homicídio")


### Tráfico de drogas

In [ ]:
# Sem vitimas, comentado para limpar o notebook

# plot_victim_incidence_by_event("Tráfico de drogas")

# plot_victim_boxplot_by_event("Tráfico de drogas")


### Pintando média e variância para queles sem ocorrência, apenas para mostrar

In [ ]:
events = ["Tráfico de drogas", "Roubo de veículo", "Roubo de carga", 
          "Roubo a instituição financeira", "Realização de vistorias", 
          "Mandado de prisão cumprido", "Furto de veículo",
          "Emissão de Alvarás de licença", "Combate a incêndios", 
          "Busca e salvamento", "Atendimento pré-hospitalar", 
          "Arma de Fogo Apreendida", "Apreensão de Maconha", 
          "Apreensão de Cocaína"]
for event in events:
    event_suffix = f" para {event}"
    with engine.connect() as conn:
        event_filter = build_event_filter(event)

        for column_name, column_label in victim_columns:
            column = getattr(pubseg.c, column_name)
            incidence_stmt = (
                select(
                    column.label("numero_vitimas"),
                    func.count().label("incidencia"),
                )
                .where(
                    column.is_not(None),
                    *event_filter,
                )
                .group_by(column)
                .order_by(column)
            )
            incidence_df = pd.read_sql(incidence_stmt, conn)

            stats = conn.execute(
                select(
                    func.avg(column).label("media"),
                    func.var_samp(column).label("variancia"),
                ).where(
                    column.is_not(None),
                    *event_filter,
                )
            ).mappings().one()

            print(f"{column_label}{event_suffix}")
            print(f"Média: {stats['media']}")
            print(f"Variância: {stats['variancia']}")
            print()
        print("================")

Masculino para Tráfico de drogas
Média: 0E-20
Variância: 0

Feminino para Tráfico de drogas
Média: 0E-20
Variância: 0

Não informado para Tráfico de drogas
Média: 0E-20
Variância: 0

Total de vítimas para Tráfico de drogas
Média: 0E-20
Variância: 0

Masculino para Roubo de veículo
Média: 0E-20
Variância: 0

Feminino para Roubo de veículo
Média: 0E-20
Variância: 0

Não informado para Roubo de veículo
Média: 0E-20
Variância: 0

Total de vítimas para Roubo de veículo
Média: 0E-20
Variância: 0

Masculino para Roubo de carga
Média: 0E-20
Variância: 0

Feminino para Roubo de carga
Média: 0E-20
Variância: 0

Não informado para Roubo de carga
Média: 0E-20
Variância: 0

Total de vítimas para Roubo de carga
Média: 0E-20
Variância: 0

Masculino para Roubo a instituição financeira
Média: 0E-20
Variância: 0

Feminino para Roubo a instituição financeira
Média: 0E-20
Variância: 0

Não informado para Roubo a instituição financeira
Média: 0E-20
Variância: 0

Total de vítimas para Roubo a instituição fi